# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo so `data/raw/...` and everything else is available. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ml-internship-assignment1
Starter data found. You're ready.


## 1. My lane as an ML task (type)

**Type: Scoring / Ranking**, with a binary classification sub-task underneath. The deliverable is a ranked review queue (scoring/ranking) — but the score itself blends a binary classifier's probability (will this page's trend flip to declining?) with a transparent rule-based score. So: ranking/scoring for the output, classification for the model doing the heavy lifting inside it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

TASK_TYPE = "Scoring / Ranking (with a classification sub-task underneath)"
print(TASK_TYPE)
print("Deliverable: a ranked review queue (scoring/ranking).")
print("Underneath: a binary classifier predicts P(page is declining), which feeds the score.")

Scoring / Ranking (with a classification sub-task underneath)
Deliverable: a ranked review queue (scoring/ranking).
Underneath: a binary classifier predicts P(page is declining), which feeds the score.


## 2. Target or proxy

**Proxy label: `is_declining_label` = (`trend_direction` == "down")**, and `trend_direction` itself is a bucket computed from `trend_pct` = (last-30-day impressions − prev-30-day impressions) / prev-30-day impressions. That's a *current-window* label, not a future observed outcome measured after a decision point — it tells me what already happened in the last 30 days, not what will happen next. I'm treating this openly as a beginner proxy label (the lane guide names it as exactly that), not the ideal capstone target. A stronger target for later weeks: prior 90 days of features → decline or recovery over the *next* 30 days, built from the warehouse's daily fact table with a real forward-looking window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())
print()

is_declining_label = (df["trend_direction"] == "down").astype(int)
print(f"is_declining_label rate: {is_declining_label.mean():.3f} ({is_declining_label.sum():,} of {len(df):,} rows)")
print()
print("trend_direction (and trend_pct, which defines it) is a CURRENT-window bucket computed from "
      "impressions_last_30d vs impressions_prev_30d -- not a future observed outcome measured after "
      "a decision point. That makes is_declining_label a PROXY label, not the ideal target. A stronger "
      "capstone label would be future-window: prior N days of features -> decline/recovery over the NEXT "
      "N days, built from the warehouse's daily fact table.")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label rate: 0.542 (16,262 of 30,000 rows)

trend_direction (and trend_pct, which defines it) is a CURRENT-window bucket computed from impressions_last_30d vs impressions_prev_30d -- not a future observed outcome measured after a decision point. That makes is_declining_label a PROXY label, not the ideal target. A stronger capstone label would be future-window: prior N days of features -> decline/recovery over the NEXT N days, built from the warehouse's daily fact table.


## 3. Success metric

**Precision@50.** The queue only matters if the top of it is right, because a reviewer works down a limited list — not the whole 30,000-row inventory. "Good" means beating the transparent baseline rule's precision@50 of 0.240 (≈12 of the top 50 correct). The starter random forest already reaches 0.740 (≈37 of 50) on this same slice, which is the bar future work has to clear or explain.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

comparison = pd.DataFrame([
    {"model": "baseline_rules", "precision_at_50": 0.240},
    {"model": "logistic_regression", "precision_at_50": 0.400},
    {"model": "decision_tree", "precision_at_50": 0.540},
    {"model": "random_forest", "precision_at_50": 0.740},
])
print(comparison)
print()
print("Metric: precision@50 -- of the top 50 pages the queue ranks first, how many are truly declining-with-demand?")
print(f"'Good' means beating the transparent baseline rule's 0.240. The starter random forest reaches "
      f"0.740: ~{round(0.740*50)} of the top 50 correct, vs ~{round(0.240*50)} for the fixed rule. "
      f"Source: outputs/model_report.md.")

                 model  precision_at_50
0       baseline_rules             0.24
1  logistic_regression             0.40
2        decision_tree             0.54
3        random_forest             0.74

Metric: precision@50 -- of the top 50 pages the queue ranks first, how many are truly declining-with-demand?
'Good' means beating the transparent baseline rule's 0.240. The starter random forest reaches 0.740: ~37 of the top 50 correct, vs ~12 for the fixed rule. Source: outputs/model_report.md.


## 4. The unit of analysis, as a real dataframe

One row = one content item (a single page) for one client, summarized over its trailing 90-day window. `content_id` + `client_id` together identify the row; loaded below with the lane-relevant columns and confirmed there are no duplicate `content_id`s (the grain holds).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

lane_cols = [
    "content_id", "client_id",
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
    "content_age_days", "days_since_last_update", "word_count",
    "trend_direction", "content_type", "position_tier", "impression_tier", "freshness_tier",
]
df = pd.read_csv("data/raw/content_refresh_anonymized.csv", usecols=lane_cols)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("One row = one (client, content item) pair: a single page's trailing-90-day snapshot.")
print(f"Grain check -- duplicate content_id rows: {df['content_id'].duplicated().sum()} (should be 0)")
print()
print(df.head())

Shape: 30,000 rows x 15 columns
One row = one (client, content item) pair: a single page's trailing-90-day snapshot.
Grain check -- duplicate content_id rows: 0 (should be 0)

             content_id          client_id     content_type  word_count  \
0  content_304f48230142  client_f369cb89fc  keyword article      3221.0   
1  content_a1fb4e703a9e  client_4e07408562  keyword article      2481.0   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article      3515.0   
3  content_331d6c4de07b  client_19581e27de  keyword article         NaN   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article      2803.0   

   impressions_90d  clicks_90d  sessions_90d  content_age_days  \
0             3803          29            17               187   
1            15320           7             9               445   
2            12581          11            11               141   
3            11751          58            78               463   
4            19140          24           14

## 5. Why ML beats a fixed rule here

The starter random forest's feature importances (below) show no single feature dominates — the top 10 share the signal, the largest at just 15.8%. That's the tell of a real but tangled pattern: recency, volume, position, age, and content depth all interact, which is exactly what one if-statement threshold can't rank well. A fixed rule already flags 58.1% of all pages as "worth a look" (see ML-02) — too blunt to say which of those to look at *first*. Weighing the signals together instead of picking one threshold, the learned model reaches precision@50 = 0.740 against the rule's 0.240 — real evidence the extra complexity earns its keep here, on this slice.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top_features = {
    "days_with_impressions": 0.1578,
    "log_impressions_90d": 0.1282,
    "avg_position": 0.1090,
    "content_age_days": 0.0955,
    "char_count": 0.0426,
    "word_count": 0.0397,
    "log_clicks_90d": 0.0346,
    "ctr": 0.0330,
    "scroll_rate": 0.0311,
    "days_with_sessions": 0.0280,
}
print("Random forest feature importances (source: outputs/model_report.md):")
for feat, imp in top_features.items():
    print(f"  {feat:<24s} {imp:.4f}")
print()
print(f"No single feature clears {max(top_features.values()):.1%} importance -- the top 10 features "
      f"together explain the decline signal, none of them alone. A fixed rule already flags 58.1% of "
      f"pages as worth a look, too blunt to prioritize within that set. The learned model, weighing all "
      f"of them together, reaches precision@50 = 0.740 vs the rule's 0.240.")

Random forest feature importances (source: outputs/model_report.md):
  days_with_impressions    0.1578
  log_impressions_90d      0.1282
  avg_position             0.1090
  content_age_days         0.0955
  char_count               0.0426
  word_count               0.0397
  log_clicks_90d           0.0346
  ctr                      0.0330
  scroll_rate              0.0311
  days_with_sessions       0.0280

No single feature clears 15.8% importance -- the top 10 features together explain the decline signal, none of them alone. A fixed rule already flags 58.1% of pages as worth a look, too blunt to prioritize within that set. The learned model, weighing all of them together, reaches precision@50 = 0.740 vs the rule's 0.240.


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.